In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder.getOrCreate()

catalog = "workspace"
silver_schema = "insurance_silver"

# Create Silver schema

spark.sql(
f"""
CREATE SCHEMA IF NOT EXISTS
{catalog}.{silver_schema}
"""
)

# Read Bronze table

insurance_df = spark.table(
    "workspace.insurance_bronze.insurance_project_dataset_raw"
)

print(
    "Bronze record count:",
    insurance_df.count()
)

In [0]:
customer_silver = (
    insurance_df
    .select(
        "customer_id",
        "customer_name",
        "gender",
        "date_of_birth",
        "city",
        "state",
        "customer_segment",
        "occupation",
        "customer_since"
    )
    .dropna(
        subset=["customer_id"]
    )
    .dropDuplicates(
        ["customer_id"]
    )
)

for column in customer_silver.columns:

    if column not in [
        "customer_id",
        "date_of_birth",
        "customer_since"
    ]:

        customer_silver = customer_silver.withColumn(
            column,
            trim(
                lower(
                    col(column).cast("string")
                )
            )
        )

customer_silver = (
    customer_silver
    .withColumn(
        "date_of_birth",
        to_date(col("date_of_birth"))
    )
    .withColumn(
        "customer_since",
        to_date(col("customer_since"))
    )
)

customer_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.insurance_silver.customer_master"
    )

print("Customer Silver table created")

In [0]:
agent_silver = (
    insurance_df
    .select(
        "agent_id",
        "agent_name",
        "branch_city",
        "agent_status"
    )
    .dropna(
        subset=["agent_id"]
    )
    .dropDuplicates(
        ["agent_id"]
    )
)

for column in agent_silver.columns:

    if column != "agent_id":

        agent_silver = agent_silver.withColumn(
            column,
            trim(
                lower(
                    col(column).cast("string")
                )
            )
        )

agent_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.insurance_silver.agent_master"
    )

print("Agent Silver table created")

In [0]:
policy_type_silver = (
    insurance_df
    .select(
        "policy_type_id",
        "policy_type"
    )
    .dropna(
        subset=["policy_type_id"]
    )
    .dropDuplicates(
        ["policy_type_id"]
    )
)

for column in policy_type_silver.columns:

    if column != "policy_type_id":

        policy_type_silver = policy_type_silver.withColumn(
            column,
            trim(
                lower(
                    col(column).cast("string")
                )
            )
        )

policy_type_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.insurance_silver.policy_type_master"
    )

print("Policy Type Silver table created")

In [0]:
policy_silver = (
    insurance_df
    .select(
        "policy_id",
        "customer_id",
        "policy_type_id",
        "agent_id",
        "policy_start_date",
        "policy_end_date",
        "policy_status",
        "sum_insured",
        "annual_premium",
        "payment_frequency",
        "policy_tenure_days",
        "outstanding_premium",
        "renewal_count",
        "latest_renewal_status"
    )
    .dropna(
        subset=[
            "policy_id",
            "customer_id",
            "policy_type_id"
        ]
    )
    .filter(
        col("sum_insured") >= 0
    )
    .filter(
        col("annual_premium") >= 0
    )
    .dropDuplicates(
        ["policy_id"]
    )
)

policy_silver = (
    policy_silver
    .withColumn(
        "policy_start_date",
        to_date(col("policy_start_date"))
    )
    .withColumn(
        "policy_end_date",
        to_date(col("policy_end_date"))
    )
)

policy_silver = policy_silver.filter(
    col("policy_end_date") >=
    col("policy_start_date")
)

for column in [
    "policy_status",
    "payment_frequency",
    "latest_renewal_status"
]:

    policy_silver = policy_silver.withColumn(
        column,
        trim(
            lower(
                col(column).cast("string")
            )
        )
    )

policy_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.insurance_silver.policy_master"
    )

print("Policy Silver table created")

In [0]:
payment_silver = (
    insurance_df
    .select(
        "payment_id",
        "payment_date",
        "payment_amount",
        "payment_status",
        "payment_method",

        "policy_id",
        "policy_type_id",
        "policy_type",
        "policy_status",
        "policy_start_date",
        "policy_end_date",
        "sum_insured",
        "annual_premium",
        "payment_frequency",
        "policy_tenure_days",
        "outstanding_premium",
        "renewal_count",
        "latest_renewal_status",

        "customer_id",
        "customer_name",
        "gender",
        "date_of_birth",
        "city",
        "state",
        "customer_segment",
        "occupation",
        "customer_since",

        "agent_id",
        "agent_name",
        "branch_city",
        "agent_status"
    )
    .dropna(
        subset=[
            "payment_id",
            "policy_id",
            "customer_id"
        ]
    )
    .filter(
        col("payment_amount") >= 0
    )
    .dropDuplicates(
        ["payment_id"]
    )
)

In [0]:
payment_silver = (
    payment_silver

    .withColumn(
        "payment_date",
        to_date(col("payment_date"))
    )

    .withColumn(
        "policy_start_date",
        to_date(col("policy_start_date"))
    )

    .withColumn(
        "policy_end_date",
        to_date(col("policy_end_date"))
    )

    .withColumn(
        "date_of_birth",
        to_date(col("date_of_birth"))
    )

    .withColumn(
        "customer_since",
        to_date(col("customer_since"))
    )
)

In [0]:
text_columns = [
    "payment_status",
    "payment_method",
    "policy_type",
    "policy_status",
    "payment_frequency",
    "latest_renewal_status",
    "customer_name",
    "gender",
    "city",
    "state",
    "customer_segment",
    "occupation",
    "agent_name",
    "branch_city",
    "agent_status"
]

for column in text_columns:

    payment_silver = payment_silver.withColumn(
        column,
        trim(
            lower(
                col(column).cast("string")
            )
        )
    )

In [0]:
target_table = "workspace.insurance_silver.insurance_payments"

payment_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(target_table)

print("Silver table created successfully")
print("Created:", target_table)
